# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to use the [mlcroissant](https://github.com/mlcommons/croissant) library to explore and process the [FAIR\² Human Protein EV Mass Spectrometry Dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa). The dataset is defined using the Croissant schema and is available via a public URL.

### Dataset Source
- Croissant Schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)
- Dataset Title: **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells**
- License: [Open Data Commons Attribution License (ODC-By) v1.0](https://opendatacommons.org/licenses/by/1-0/)


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display core metadata
meta = dataset.metadata
print(f"Dataset Name: {meta.name}")
print(f"Description: {meta.description}")
print(f"Published: {getattr(meta, 'datePublished', None)}")
print(f"Version: {getattr(meta, 'version', None)}")
print(f"License: {getattr(meta, 'license', None)}")
print(f"Authors: {getattr(meta, 'author', None)}")

## 2. Data Overview

Review available *record sets* and describe the fields for each, referencing records, fields, and columns by their `@id` as per the Croissant schema.


In [ ]:
# List all record sets and their corresponding @id

record_sets = list(dataset.record_sets)
print(f"Number of Record Sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    name = rs.get('name', '(no name)')
    desc = rs.get('description', '(no description)')
    print(f" - Name: {name}")
    print(f" - Description: {desc}")
    print(f" - Fields:")
    if 'field' in rs:
        for field in rs['field']:
            # Field can be '@id' string or field dict
            field_id = field['@id'] if isinstance(field, dict) and '@id' in field else field
            print(f"    * Field @id: {field_id}")
    print("")
if len(record_sets) == 0:
    print("No record sets were directly listed in the schema; mlcroissant may have inferred them from distribution/data files.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for further analysis. Reference record set by its `@id`.


In [ ]:
# List all record set @ids and load data for each
rs_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}

for rs_id in rs_ids:
    try:
        print(f"Loading records for record set: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  -> Columns: {df.columns.tolist()}")
        print(f"  -> Number of rows: {len(df)}\n")
    except Exception as e:
        print(f"  -> Failed to load {rs_id}: {e}")

# Example: Print columns and preview for first record set
if rs_ids:
    main_rs_id = rs_ids[0]
    print(f"\nFields for record set {main_rs_id}:")
    if main_rs_id in dataframes:
        df = dataframes[main_rs_id]
        print(df.head())

## 4. Exploratory Data Analysis (EDA)

Let's select a *numeric field* (e.g., 'coverage', 'peptide_count', 'MW', 'abundance_mean' or similar depending on the actual columns) to demonstrate filtering, normalization, and grouping. All field and column variables are referenced explicitly by their schema `@id` (or derived DataFrame column name as imported by `mlcroissant`).


In [ ]:
# Pick a numeric field for analysis. Replace below with an actual field name (use columns printed above).

main_rs_id = rs_ids[0] if rs_ids else None
numeric_field = None
group_field = None

if main_rs_id and main_rs_id in dataframes:
    df = dataframes[main_rs_id]
    potential_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print("Possible numeric fields detected:", potential_numeric_fields)
    # Fallback: pick first
    if potential_numeric_fields:
        numeric_field = potential_numeric_fields[0]
    # Try and pick a good group-by field
    potential_group_fields = [c for c in df.columns if c not in potential_numeric_fields and df[c].nunique() < max(10, len(df)//10)]
    if potential_group_fields:
        group_field = potential_group_fields[0]

if numeric_field is not None:
    print(f"Using numeric_field: {numeric_field}")
    threshold = df[numeric_field].mean() if not pd.isnull(df[numeric_field].mean()) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"\nFiltered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df[[numeric_field]].head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field is not None and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization

Visualize the distribution of the selected numeric field and the normalized output. If a group field exists, plot group means as well.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and main_rs_id in dataframes and numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field], kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
    
    if group_field is not None and group_field in df.columns:
        plt.figure(figsize=(10, 4))
        group_stats = df.groupby(group_field)[numeric_field].mean().reset_index()
        sns.barplot(x=group_field, y=numeric_field, data=group_stats)
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(f'Mean {numeric_field}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we used the `mlcroissant` library to explore the FAIR\² human protein mass spectrometry dataset via its Croissant metadata schema. We examined the available record sets, loaded a data table, and performed basic filtering, normalization, and visualization using numeric fields. For further bioinformatics or proteomics analysis, cross-reference the schema `@id` values, available columns, and scientific context in the original FAIR\² [data documentation](https://sen.science/doi/10.71728/senscience.vq4a-28xa) or publication.